# EvalHub Client — Local LightEval Evaluation

This notebook walks through the full evaluation lifecycle using EvalHub in local mode: connecting to the server, submitting an evaluation job, monitoring progress, and inspecting results in MLflow and the OCI registry.

**Steps:**

1. **Connect to EvalHub** — verify the server is healthy
2. **Discover providers and benchmarks** — list what is available for evaluation
3. **Verify model availability** — confirm the LLM endpoint is serving the target model
4. **Submit an evaluation job** — configure and submit a GSM8K benchmark run
5. **Monitor job progress** — poll until the job completes or fails
6. **Inspect results** — review the job response, MLflow experiment runs, and OCI artifacts
7. **Cleanup** — delete the job from the server

> **Kernel:** Select `.venv/bin/python` as the Jupyter kernel.
> **Prerequisites:** See `readme.md` for the setup required before running this notebook.

In [1]:
from evalhub import (
    SyncEvalHubClient,
    JobSubmissionRequest,
    BenchmarkConfig,
    ModelConfig,
    JobStatus,
    ExperimentConfig,
    EvaluationExports,
    EvaluationExportsOCI,
    OCICoordinates,
)
import os

## 1. Connect to EvalHub

In [2]:
evalhub_url = "http://localhost:8080"
client = SyncEvalHubClient(base_url=evalhub_url)
health = client.health()
print(f"✓ EvalHub is healthy: {health['status']}")

✓ EvalHub is healthy: healthy


## 2. Discover providers and benchmarks

### 2.1 List providers

In [3]:
providers = client.providers.list()
print(f"✓ Found {len(providers)} providers:")
for provider in providers:
    print(f"  - {provider.resource.id}: {provider.name}")

✓ Found 1 providers:
  - lighteval: LightEval


### 2.2 List benchmarks

In [4]:
provider_id = "lighteval"

In [5]:
benchmarks = client.benchmarks.list(provider_id=provider_id)
print(f"\n✓ Found {len(benchmarks)} benchmarks for {provider_id}:")
for benchmark in benchmarks:
    print(f"  - {benchmark.id}: {benchmark.name}")


✓ Found 1 benchmarks for lighteval:
  - gsm8k: Grade-school math word problems


## 3. Verify model availability

In [6]:
model_url = "http://localhost:11434/v1"
model_name = "llama3.2:3b-instruct-q4_K_M"

In [7]:
import requests

response = requests.get(f"{model_url}/models")
models = response.json()
model_info = next(m for m in models["data"] if m["id"] == model_name)
print(model_info)

{'id': 'llama3.2:3b-instruct-q4_K_M', 'object': 'model', 'created': 1778599771, 'owned_by': 'library'}


## 4. Submit an evaluation job

Configure the job parameters — model, benchmark, experiment tracking, and OCI export — then submit.

In [8]:
# evalhub
job_name = "ev-job1"
benchmark_id = "gsm8k"

# mlflow
experiment_name = "ev-exp1"

# oci
oci_host = "localhost:5001"
oci_repository = "eval-results"

In [9]:
from evalhub import ExperimentTag

job_request = JobSubmissionRequest(
    name=job_name,
    model=ModelConfig(url=model_url, name=model_name),
    benchmarks=[
        BenchmarkConfig(
            id=benchmark_id,
            provider_id=provider_id,
            parameters={
                "num_examples": 10,  # Number of examples to evaluate
                "num_few_shot": 0,
            },
        )
    ],
    experiment=ExperimentConfig(
        name=experiment_name,
        description=f"Local LightEval evaluation experiment for {model_name}",
        tags=[
            ExperimentTag(key="model", value=model_name),
            ExperimentTag(key="provider_id", value=provider_id),
        ],
    ),
    exports=EvaluationExports(
        oci=EvaluationExportsOCI(
            coordinates=OCICoordinates(
                oci_host=oci_host,
                oci_repository=oci_repository,
                annotations={
                    "job_name": job_name,
                    "experiment_name": experiment_name,
                },
            )
        )
    ),
)

# Submit job using nested resource
job = client.jobs.submit(job_request)

# Store job ID for later use
job_id = job.id
print(f"✓ Job submitted successfully")
print(f"  Job ID: {job_id}")
print(f"  State: {job.state}")
print(f"  Created: {job.resource.created_at}\n")
print(job.model_dump_json(indent=2))

✓ Job submitted successfully
  Job ID: 9c9cc498-0a18-419c-9026-f700b538e8da
  State: JobStatus.PENDING
  Created: 2026-05-13 14:01:56.835518+05:30

{
  "resource": {
    "id": "9c9cc498-0a18-419c-9026-f700b538e8da",
    "tenant": null,
    "created_at": "2026-05-13T14:01:56.835518+05:30",
    "updated_at": null,
    "mlflow_experiment_id": "1",
    "message": null
  },
  "status": {
    "state": "pending",
    "message": {
      "message": "Evaluation job created",
      "message_code": "evaluation_job_created"
    },
    "benchmarks": []
  },
  "results": {
    "benchmarks": [],
    "mlflow_experiment_url": "http://localhost:5000/api/2.0/mlflow/experiments"
  },
  "name": "ev-job1",
  "description": null,
  "tags": [],
  "model": {
    "url": "http://localhost:11434/v1",
    "name": "llama3.2:3b-instruct-q4_K_M",
    "auth": null
  },
  "benchmarks": [
    {
      "id": "gsm8k",
      "provider_id": "lighteval",
      "parameters": {
        "num_examples": 10,
        "num_few_shot":

## 5. Monitor job progress

Poll the job status until it completes or fails.

In [10]:
import time

updated_job = client.jobs.get(job_id=job_id)

while updated_job.state not in (JobStatus.COMPLETED, JobStatus.FAILED):
    print(f"\n✓ Current job state: {updated_job.state}")
    time.sleep(5)
    updated_job = client.jobs.get(job_id=job_id)

if updated_job.state == JobStatus.FAILED:
    if updated_job.status and updated_job.status.message:
        print(f"Job failed: {updated_job.status.message.message}")
    else:
        print("Job failed: Unknown")
else:
    print(f"✓ Job completed successfully: {updated_job.state}")


✓ Current job state: JobStatus.RUNNING

✓ Current job state: JobStatus.RUNNING

✓ Current job state: JobStatus.RUNNING

✓ Current job state: JobStatus.RUNNING

✓ Current job state: JobStatus.RUNNING

✓ Current job state: JobStatus.RUNNING
✓ Job completed successfully: JobStatus.COMPLETED


## 6. Inspect results

### 6.1 Completed job response

In [12]:
print(updated_job.model_dump_json(indent=2))

{
  "resource": {
    "id": "9c9cc498-0a18-419c-9026-f700b538e8da",
    "tenant": null,
    "created_at": "2026-05-13T08:31:56Z",
    "updated_at": "2026-05-13T08:32:25Z",
    "mlflow_experiment_id": "1",
    "message": null
  },
  "status": {
    "state": "completed",
    "message": {
      "message": "Evaluation job is completed",
      "message_code": "evaluation_job_updated"
    },
    "benchmarks": [
      {
        "id": "gsm8k",
        "provider_id": "lighteval",
        "benchmark_index": 0,
        "status": "completed",
        "error_message": null,
        "started_at": null,
        "completed_at": "2026-05-13T08:32:24.897553Z"
      }
    ]
  },
  "results": {
    "benchmarks": [
      {
        "id": "gsm8k",
        "provider_id": "lighteval",
        "benchmark_index": 0,
        "metrics": {
          "all.extractive_match": 0.7,
          "gsm8k|0.extractive_match": 0.7
        },
        "artifacts": {
          "evalhub.env_card": {
            "aggregate_results"

### 6.2 MLflow experiment runs

Query MLflow for the experiment runs created by the evaluation job.

In [13]:
import mlflow

mlflow_url = "http://localhost:5000"

mlflow.set_tracking_uri(mlflow_url)
runs = mlflow.search_runs(search_all_experiments=True)

print(runs.columns.tolist())

['run_id', 'experiment_id', 'status', 'artifact_uri', 'start_time', 'end_time', 'metrics.all.extractive_match', 'metrics.gsm8k_0.extractive_match', 'metrics.overall_score', 'params.model_name', 'params.num_examples_evaluated', 'params.duration_seconds', 'params.benchmark_id', 'tags.mlflow.runName', 'tags.provider_id', 'tags.model']


In [14]:
runs

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.all.extractive_match,metrics.gsm8k_0.extractive_match,metrics.overall_score,params.model_name,params.num_examples_evaluated,params.duration_seconds,params.benchmark_id,tags.mlflow.runName,tags.provider_id,tags.model
0,41d4fb940bf743d384c2f12ab891abe1,1,FINISHED,mlflow-artifacts:/1/41d4fb940bf743d384c2f12ab8...,2026-05-13 08:32:24.956000+00:00,2026-05-13 08:32:24.968000+00:00,0.7,0.7,0.7,llama3.2:3b-instruct-q4_K_M,10,27.666032075881958,gsm8k,9c9cc498-0a18-419c-9026-f700b538e8da,lighteval,llama3.2:3b-instruct-q4_K_M


In [15]:
cols = [
    c
    for c in runs.columns
    if c.startswith(("metrics.", "params."))
    or c in ("run_id")
]
runs[cols].head()

,run_id,metrics.all.extractive_match,metrics.gsm8k_0.extractive_match,metrics.overall_score,params.model_name,params.num_examples_evaluated,params.duration_seconds,params.benchmark_id
0,41d4fb940bf743d384c2f12ab891abe1,0.7,0.7,0.7,llama3.2:3b-instruct-q4_K_M,10,27.666032075881958,gsm8k


### 6.3 OCI artifacts

Verify that the evaluation results were pushed to the OCI registry.

In [16]:
oci_reference = updated_job.results.benchmarks[0].artifacts["oci_reference"]
print(f"oci_reference: {oci_reference}")

oci_reference: localhost:5001/eval-results:evalhub-b5c38938e0f23bb70ee4b125d52f5e3b55a23abd710f08a719213de593f7a233@sha256:34e99e53297e41601124256eabd123b721084b380ca63b64ccec7b28f084f078


In [17]:
from oras.client import OrasClient

oras_client = OrasClient(insecure=True)

# Fetch manifest for the experiment tag
manifest = oras_client.get_manifest(oci_reference)
import pprint
pprint.pprint(manifest)

{'annotations': {'experiment_name': 'ev-exp1',
                 'io.github.eval-hub.benchmark': 'gsm8k',
                 'io.github.eval-hub.benchmark_id': 'gsm8k',
                 'io.github.eval-hub.job_id': '9c9cc498-0a18-419c-9026-f700b538e8da',
                 'io.github.eval-hub.model': 'llama3.2:3b-instruct-q4_K_M',
                 'io.github.eval-hub.provider_id': 'lighteval',
                 'job_name': 'ev-job1',
                 'org.opencontainers.image.created': '2026-05-13T08:32:24.820805+00:00'},
 'artifactType': 'application/vnd.eval-hub.github.io',
 'config': {'digest': 'sha256:90977d9e890e3322e05811e768da4681520956cd097030fcd797c46b77ff0ee3',
            'mediaType': 'application/vnd.oci.image.config.v1+json',
            'size': 609},
 'layers': [{'annotations': {'olot.layer.content.name': 'lighteval_results.json'},
             'digest': 'sha256:2f10dc978de7cc8435948bae0f21d0fe84d77123ea02b8736ebd256554b11796',
             'mediaType': 'application/vnd.oci.ima

#### 6.3.1 Inspect layer contents

In [19]:
import tarfile
import io
import json

for i, layer in enumerate(manifest["layers"], start=1):
    name = layer.get("annotations", {}).get("olot.layer.content.name", "unknown")
    response = oras_client.get_blob(f"{oci_host}/{oci_repository}", layer["digest"])

    print(f"=== {i}. {name} ===")
    try:
        tar = tarfile.open(fileobj=io.BytesIO(response.content))
        for member in tar.getmembers():
            f = tar.extractfile(member)
            if f:
                content = f.read().decode("utf-8")
                if member.name.endswith(".json"):
                    print(json.dumps(json.loads(content), indent=2))
                else:
                    print(content)
        tar.close()
    except tarfile.TarError:
        print(response.text)
    print()

=== 1. lighteval_results.json ===
{
  "config_general": {
    "lighteval_sha": "?",
    "num_fewshot_seeds": 1,
    "max_samples": 10,
    "job_id": "0",
    "start_time": 571903.2454825,
    "end_time": 571911.55315025,
    "total_evaluation_time_secondes": "8.307667750050314",
    "model_config": {
      "model_name": "openai/llama3.2:3b-instruct-q4_K_M",
      "generation_parameters": {
        "num_blocks": null,
        "block_size": null,
        "early_stopping": null,
        "repetition_penalty": null,
        "frequency_penalty": null,
        "length_penalty": null,
        "presence_penalty": null,
        "max_new_tokens": null,
        "min_new_tokens": null,
        "seed": null,
        "stop_tokens": null,
        "temperature": 0,
        "top_k": null,
        "min_p": null,
        "top_p": null,
        "truncate_prompt": null,
        "cache_implementation": null,
        "response_format": null
      },
      "system_prompt": null,
      "cache_dir": "~/.cache/hu

## 7. Cleanup

In [20]:
client.jobs.cancel(job_id=job_id, hard_delete=True)

True